# CIVA — Notebook Perbaikan Dataset Buku (`dataset_v3`)

Notebook ini menyiapkan proses peningkatan kualitas dataset buku sebelum training ulang CIVA.

**Prinsip penting**
- File sumber tidak ditimpa.
- Normalisasi teks untuk pemeriksaan tidak otomatis mengubah identitas bibliografi.
- Variasi urutan nama penulis tidak langsung dianggap sebagai mismatch/halu.
- Data mismatch yang belum diverifikasi hanya menjadi *review queue*, bukan langsung dimasukkan sebagai label halu final.
- Dataset final hanya dibentuk dari data yang status review-nya sudah ditetapkan.

## Tujuan
1. Memeriksa kualitas metadata buku valid.
2. Menemukan duplikasi dan anomali penulis tanpa menghapus informasi secara agresif.
3. Menyiapkan antrean pemeriksaan untuk data mismatch.
4. Menghasilkan file staging yang aman untuk ditinjau sebelum digabungkan ke dataset CIVA.

Jalankan notebook dari folder utama repository `D:\capstone\civa` agar struktur folder relatif terdeteksi.

## 0. Konfigurasi dan lokasi file

Sesuaikan `PROJECT_ROOT` jika notebook tidak dijalankan dari folder utama repository. Notebook mencari file sumber di `dataset/additional_books/` atau `dataset_v2/additional_books/`, dan juga memeriksa folder kerja saat ini.

In [1]:
from pathlib import Path
import json
import re
import unicodedata
import pandas as pd
import numpy as np
from IPython.display import display

# Jika notebook dijalankan dari D:\capstone\civa, biarkan None.
# Jika tidak, isi dengan Path(r"D:\capstone\civa").
PROJECT_ROOT = None

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path(PROJECT_ROOT)

VALID_FILENAME = "TAMBAHAN_data_buku_valid_5394.csv"
MISMATCH_FILENAME = "TAMBAHAN_mismatch_buku_2kolom.csv"

candidate_dirs = [
    PROJECT_ROOT / "dataset" / "additional_books",
    PROJECT_ROOT / "dataset_v2" / "additional_books",
    PROJECT_ROOT / "additional_books",
    PROJECT_ROOT,
]

def locate_file(filename):
    for folder in candidate_dirs:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{filename} tidak ditemukan. Periksa PROJECT_ROOT dan letak file."
    )

VALID_PATH = locate_file(VALID_FILENAME)
MISMATCH_PATH = locate_file(MISMATCH_FILENAME)

OUTPUT_DIR = PROJECT_ROOT / "dataset_v3_staging"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT.resolve())
print("Valid source :", VALID_PATH.resolve())
print("Mismatch source:", MISMATCH_PATH.resolve())
print("Output       :", OUTPUT_DIR.resolve())

Project root : D:\capstone\civa
Valid source : D:\capstone\civa\dataset\additional_books\TAMBAHAN_data_buku_valid_5394.csv
Mismatch source: D:\capstone\civa\dataset\additional_books\TAMBAHAN_mismatch_buku_2kolom.csv
Output       : D:\capstone\civa\dataset_v3_staging


## 1. Membaca data dan memeriksa skema

Tahap ini hanya membaca file sumber. Tidak ada perubahan yang disimpan ke file asli.

In [2]:
valid_raw = pd.read_csv(
    VALID_PATH, encoding="utf-8-sig", keep_default_na=False
)
mismatch_raw = pd.read_csv(
    MISMATCH_PATH, encoding="utf-8-sig", keep_default_na=False
)

print("VALID BOOKS:", valid_raw.shape)
print("Kolom:", valid_raw.columns.tolist())
display(valid_raw.head())

print("\nMISMATCH CANDIDATES:", mismatch_raw.shape)
print("Kolom:", mismatch_raw.columns.tolist())
display(mismatch_raw.head())

required_valid = ["judul", "author"]
required_mismatch = ["judul", "author"]

assert all(c in valid_raw.columns for c in required_valid), "Kolom wajib valid tidak lengkap."
assert all(c in mismatch_raw.columns for c in required_mismatch), "Kolom wajib mismatch tidak lengkap."

print("\nStatus pembacaan: OK")

VALID BOOKS: (5894, 11)
Kolom: ['id', 'sumber_api', 'label_asli', 'jenis_referensi', 'judul', 'link_referensi', 'tahun', 'author', 'scope', 'bidang', 'bahasa']


,id,sumber_api,label_asli,jenis_referensi,judul,link_referensi,tahun,author,scope,bidang,bahasa
0,6000,OpenLibrary,1,book,Computer science,https://openlibrary.org/works/OL3275955W,1985,J. Glenn Brookshear; Brookshear,cs,Computer Science,eng
1,6001,OpenLibrary,1,book,Invitation to Computer Science,https://openlibrary.org/works/OL1926026W,1995,G. Michael Schneider; Judith L. Gersting; Judi...,cs,Computer Science,eng
2,6002,OpenLibrary,1,book,Hackers,https://openlibrary.org/works/OL804638W,1984,Steven Levy,cs,Computer Science,tur
3,6003,OpenLibrary,1,book,"Advances in Computers, Volume 49 (Advances in ...",https://openlibrary.org/works/OL7938163W,1995,Marvin V. Zelkowitz; Marshall C. Yovits,cs,Computer Science,eng
4,6004,OpenLibrary,1,book,Computer science,https://openlibrary.org/works/OL35529W,1997,Behrouz A. Forouzan; Richard F. Gilberg,cs,Computer Science,eng



MISMATCH CANDIDATES: (1713, 2)
Kolom: ['judul', 'author']


,judul,author
0,Britannica,deLange et Reeuwijk van al Feijs
1,Working with Microsoft Visual studio 2005 team...,Hundhausen Richard
2,Optimization Theory Based on Neutrosophic and ...,Mohamed Abdel-Baset; Florentin Smarandache
3,Socioeconomic and legal implications of electr...,Politis Dionysios
4,"Textbook of orthopaedic medicine, v. 2",Henry James Cyriax



Status pembacaan: OK


## 2. Audit kualitas data buku valid

Audit mencakup:
- nilai judul/author kosong;
- spasi berlebih dan karakter yang perlu dinormalisasi;
- duplikasi pasangan judul-author secara tekstual;
- duplikasi judul dengan author berbeda;
- nama author yang berulang dalam satu sel.

**Penting:** nama penulis tidak otomatis dibalik atau ditulis ulang. Contoh `David W. Snoke` dan `Snoke W David` memerlukan pemeriksaan identitas; notebook tidak akan menganggap keduanya pasti sama.

In [3]:
def basic_text(value):
    value = unicodedata.normalize("NFKC", "" if value is None else str(value))
    value = value.replace("\ufffd", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value

def title_key(value):
    # Kunci pembanding, bukan pengganti judul asli.
    value = basic_text(value).lower()
    value = value.strip(" \t\r\n\"'“”‘’«».,;:-–—")
    return value

def author_token_key(value):
    # Normalisasi ringan untuk mendeteksi duplikasi author persis.
    # Tidak menyamakan "David W Snoke" dengan "Snoke W David".
    value = basic_text(value).lower()
    value = re.sub(r"[.]", "", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

valid = valid_raw.copy()

for col in ["judul", "author", "link_referensi", "sumber_api", "scope", "bidang", "bahasa"]:
    if col in valid.columns:
        valid[col] = valid[col].map(basic_text)

valid["title_key"] = valid["judul"].map(title_key)
valid["author_key"] = valid["author"].map(author_token_key)

valid["judul_kosong"] = valid["title_key"].eq("")
valid["author_kosong"] = valid["author_key"].eq("")

# Duplikasi pasangan tekstual, bukan identitas bibliografi yang dipastikan.
valid["duplikat_pasangan_teks"] = valid.duplicated(
    subset=["title_key", "author_key"], keep=False
)

valid["judul_sama_author_berbeda"] = valid.duplicated(
    subset=["title_key"], keep=False
) & ~valid["duplikat_pasangan_teks"]

def repeated_author_names(author_text):
    parts = [basic_text(x) for x in re.split(r"\s*[;|]\s*", str(author_text))]
    parts = [x for x in parts if x]
    keys = [author_token_key(x) for x in parts]
    return len(keys) != len(set(keys))

valid["author_duplikat_dalam_sel"] = valid["author"].map(repeated_author_names)

audit_summary = pd.Series({
    "jumlah_baris": len(valid),
    "judul_kosong": int(valid["judul_kosong"].sum()),
    "author_kosong": int(valid["author_kosong"].sum()),
    "duplikat_pasangan_teks": int(valid["duplikat_pasangan_teks"].sum()),
    "judul_sama_author_berbeda": int(valid["judul_sama_author_berbeda"].sum()),
    "author_duplikat_dalam_sel": int(valid["author_duplikat_dalam_sel"].sum()),
})

display(audit_summary.to_frame("jumlah"))


,jumlah
jumlah_baris,5894
judul_kosong,0
author_kosong,0
duplikat_pasangan_teks,0
judul_sama_author_berbeda,569
author_duplikat_dalam_sel,53


## 3. Tinjau kandidat masalah metadata

Lihat sampel kasus sebelum menentukan tindakan. Jangan menghapus semua judul yang sama: edisi berbeda atau karya berbeda dapat memiliki judul serupa.

In [4]:
print("A. Baris dengan judul/author kosong")
display(valid[valid["judul_kosong"] | valid["author_kosong"]].head(30))

print("B. Duplikasi pasangan judul-author secara tekstual")
display(
    valid.loc[valid["duplikat_pasangan_teks"],
              [c for c in ["id", "judul", "author", "tahun", "link_referensi", "scope"] if c in valid.columns]]
    .sort_values("judul")
    .head(50)
)

print("C. Judul sama tetapi author berbeda")
display(
    valid.loc[valid["judul_sama_author_berbeda"],
              [c for c in ["id", "judul", "author", "tahun", "link_referensi", "scope"] if c in valid.columns]]
    .sort_values("judul")
    .head(50)
)

print("D. Author berulang dalam satu sel")
display(
    valid.loc[valid["author_duplikat_dalam_sel"],
              [c for c in ["id", "judul", "author", "link_referensi"] if c in valid.columns]]
    .head(50)
)

A. Baris dengan judul/author kosong


,id,sumber_api,label_asli,jenis_referensi,judul,link_referensi,tahun,author,scope,bidang,bahasa,title_key,author_key,judul_kosong,author_kosong,duplikat_pasangan_teks,judul_sama_author_berbeda,author_duplikat_dalam_sel


B. Duplikasi pasangan judul-author secara tekstual


,id,judul,author,tahun,link_referensi,scope


C. Judul sama tetapi author berbeda


,id,judul,author,tahun,link_referensi,scope
1009,7009,A textbook of economics,J. L. Hanson,1977,https://openlibrary.org/works/OL9220656W,non_cs
1061,7061,A textbook of economics,Frank Livesey,1978,https://openlibrary.org/works/OL24173167W,non_cs
1167,7167,Abnormal Psychology,Sarason,1998,https://openlibrary.org/works/OL8812060W,non_cs
5390,11390,Abnormal psychology,James H. Hansell; Lisa K. Damour,2008,https://openlibrary.org/works/OL18662548W,non_cs
952,6952,Abnormal psychology,David H. Barlow; V. Mark Durand; David H. Barl...,1995,https://openlibrary.org/works/OL1885798W,non_cs
468,6468,Accounting Information Systems,Leslie Turner; Andrea Weickgenannt,2008,https://openlibrary.org/works/OL20761457W,cs
439,6439,Accounting information systems,George H. Bodnar; William S. Hopwood,1980,https://openlibrary.org/works/OL2722745W,cs
498,6498,Accounting information systems,James Richard Davis; Barry E. Cushing; James D...,1979,https://openlibrary.org/works/OL19142311W,cs
725,6725,Addison-Wesley Mathematics,Robert E. Eicholz; Phares G. O'Daffer; Randall...,1985,https://openlibrary.org/works/OL15132203W,non_cs
5218,11218,Addison-Wesley mathematics,Addison-Wesley Publishing Company,1991,https://openlibrary.org/works/OL15396144W,non_cs


D. Author berulang dalam satu sel


,id,judul,author,link_referensi
37,6037,Computational fluid dynamics,Jiyuan Tu; Jiyuan TU; Guan Heng YEOH; Chaoqun LIU,https://openlibrary.org/works/OL13757864W
64,6064,How computers work,Ron White; Ron White; Timothy Edward Downs,https://openlibrary.org/works/OL11337267W
108,6108,Software testing,"Gerald D. Everett; Gerald D. Everett; Raymond,...",https://openlibrary.org/works/OL12220003W
358,6358,Slot Machine,Chris Lynch; Chris Lynch,https://openlibrary.org/works/OL1921233W
392,6392,Astronomical Almanac for the Year 1993,Rand McNally; United States Naval Observatory;...,https://openlibrary.org/works/OL166256W
416,6416,Data structures,Richard F. Gilberg; Richard F. Gilberg; Behrou...,https://openlibrary.org/works/OL13756190W
456,6456,Object-oriented systems analysis and design us...,Simon Bennett; Simon Bennett; Steve McRobb; Ra...,https://openlibrary.org/works/OL12743268W
475,6475,Geographic information systems,Desmond Ball; Ross Babbage; Desmond Ball,https://openlibrary.org/works/OL18866912W
543,6543,Nonlinear dynamics and Chaos,Steven H. Strogatz; Steven Strogatz; Steven H....,https://openlibrary.org/works/OL3907581W
581,6581,Chemistry,Theodore L. Brown; H. Eugene Lemay; Bruce E. B...,https://openlibrary.org/works/OL15130687W


## 4. Membuat versi bersih yang konservatif

Yang dilakukan otomatis hanya:
- merapikan whitespace dan Unicode pada kolom teks;
- menghapus baris yang judulnya kosong;
- menghapus duplikasi **pasangan judul-author yang sama secara tekstual**, dengan mempertahankan kemunculan pertama.

Yang **tidak** dilakukan:
- membalik urutan nama penulis;
- menghapus penulis hanya karena namanya mirip;
- menghapus judul sama dengan author berbeda;
- mengubah tahun atau metadata sumber;
- menganggap data yang tidak punya author pasti tidak valid.

Periksa hasil dan jumlah baris yang dikeluarkan sebelum dipakai.

In [5]:
# Kandidat valid yang bisa dipakai setelah pemeriksaan.
valid_clean = valid.copy()

# Judul kosong tidak bisa dipakai oleh model judul+author.
valid_clean = valid_clean[~valid_clean["judul_kosong"]].copy()

# Buang hanya duplikasi pasangan teks yang identik.
valid_clean = valid_clean.drop_duplicates(
    subset=["title_key", "author_key"],
    keep="first"
).copy()

# Status valid mengikuti sumber asli, tetapi hanya jika label_asli memang bernilai 1.
if "label_asli" in valid_clean.columns:
    valid_clean["status"] = pd.to_numeric(
        valid_clean["label_asli"], errors="coerce"
    )
else:
    valid_clean["status"] = 1

valid_clean = valid_clean[valid_clean["status"] == 1].copy()
valid_clean["jenis"] = "asli"
valid_clean["review_status"] = "VALID_FROM_SOURCE_PENDING_QA"

print("Baris sumber       :", len(valid_raw))
print("Baris setelah bersih:", len(valid_clean))
print("Baris dikeluarkan  :", len(valid_raw) - len(valid_clean))

display(valid_clean.head())

Baris sumber       : 5894
Baris setelah bersih: 5894
Baris dikeluarkan  : 0


,id,sumber_api,label_asli,jenis_referensi,judul,link_referensi,tahun,author,scope,bidang,...,title_key,author_key,judul_kosong,author_kosong,duplikat_pasangan_teks,judul_sama_author_berbeda,author_duplikat_dalam_sel,status,jenis,review_status
0,6000,OpenLibrary,1,book,Computer science,https://openlibrary.org/works/OL3275955W,1985,J. Glenn Brookshear; Brookshear,cs,Computer Science,...,computer science,j glenn brookshear; brookshear,False,False,False,True,False,1,asli,VALID_FROM_SOURCE_PENDING_QA
1,6001,OpenLibrary,1,book,Invitation to Computer Science,https://openlibrary.org/works/OL1926026W,1995,G. Michael Schneider; Judith L. Gersting; Judi...,cs,Computer Science,...,invitation to computer science,g michael schneider; judith l gersting; judith...,False,False,False,False,False,1,asli,VALID_FROM_SOURCE_PENDING_QA
2,6002,OpenLibrary,1,book,Hackers,https://openlibrary.org/works/OL804638W,1984,Steven Levy,cs,Computer Science,...,hackers,steven levy,False,False,False,False,False,1,asli,VALID_FROM_SOURCE_PENDING_QA
3,6003,OpenLibrary,1,book,"Advances in Computers, Volume 49 (Advances in ...",https://openlibrary.org/works/OL7938163W,1995,Marvin V. Zelkowitz; Marshall C. Yovits,cs,Computer Science,...,"advances in computers, volume 49 (advances in ...",marvin v zelkowitz; marshall c yovits,False,False,False,False,False,1,asli,VALID_FROM_SOURCE_PENDING_QA
4,6004,OpenLibrary,1,book,Computer science,https://openlibrary.org/works/OL35529W,1997,Behrouz A. Forouzan; Richard F. Gilberg,cs,Computer Science,...,computer science,behrouz a forouzan; richard f gilberg,False,False,False,True,False,1,asli,VALID_FROM_SOURCE_PENDING_QA


## 5. Menyiapkan antrean review mismatch

File mismatch berisi dua kolom (`judul`, `author`) dan tidak menyertakan sumber buku asal atau bukti bibliografis. Karena itu, notebook **tidak boleh menganggap seluruh 1.713 baris otomatis merupakan halu yang benar**.

Setiap baris diberi status `NEEDS_MANUAL_VERIFICATION`. Isi kolom `review_status` setelah memeriksa sumber:
- `CONFIRMED_HALU`: pasangan judul-author memang tidak cocok / hasil manipulasi yang memang ingin diuji.
- `CONFIRMED_VALID`: ternyata hanya variasi format atau masih merupakan referensi yang sah.
- `REJECT_MAPPING`: pasangan sumber atau konstruksi contoh tidak dapat dipastikan.
- `NEEDS_MANUAL_VERIFICATION`: belum diperiksa.

Untuk menilai mismatch, gunakan bukti bibliografis yang tersedia (tautan katalog, penerbit, ISBN, atau rekaman sumber). Jangan menentukan label hanya dari similarity score.

In [6]:
mismatch_review = mismatch_raw.copy()

mismatch_review["judul"] = mismatch_review["judul"].map(basic_text)
mismatch_review["author"] = mismatch_review["author"].map(basic_text)

mismatch_review.insert(
    0, "review_id",
    [f"MM-{i:05d}" for i in range(1, len(mismatch_review) + 1)]
)

mismatch_review["status_candidate"] = 0
mismatch_review["jenis_candidate"] = "mismatch_buku"
mismatch_review["review_status"] = "NEEDS_MANUAL_VERIFICATION"
mismatch_review["review_notes"] = ""

# Kunci untuk menemukan duplikasi internal pada daftar mismatch.
mismatch_review["title_key"] = mismatch_review["judul"].map(title_key)
mismatch_review["author_key"] = mismatch_review["author"].map(author_token_key)

mismatch_review["duplikat_pasangan_teks"] = mismatch_review.duplicated(
    subset=["title_key", "author_key"], keep=False
)

print("Jumlah kandidat mismatch:", len(mismatch_review))
print("Duplikasi pasangan teks:", int(mismatch_review["duplikat_pasangan_teks"].sum()))

display(mismatch_review.head(20))

Jumlah kandidat mismatch: 1713
Duplikasi pasangan teks: 0


,review_id,judul,author,status_candidate,jenis_candidate,review_status,review_notes,title_key,author_key,duplikat_pasangan_teks
0,MM-00001,Britannica,deLange et Reeuwijk van al Feijs,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,britannica,delange et reeuwijk van al feijs,False
1,MM-00002,Working with Microsoft Visual studio 2005 team...,Hundhausen Richard,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,working with microsoft visual studio 2005 team...,hundhausen richard,False
2,MM-00003,Optimization Theory Based on Neutrosophic and ...,Mohamed Abdel-Baset; Florentin Smarandache,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,optimization theory based on neutrosophic and ...,mohamed abdel-baset; florentin smarandache,False
3,MM-00004,Socioeconomic and legal implications of electr...,Politis Dionysios,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,socioeconomic and legal implications of electr...,politis dionysios,False
4,MM-00005,"Textbook of orthopaedic medicine, v. 2",Henry James Cyriax,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,"textbook of orthopaedic medicine, v. 2",henry james cyriax,False
5,MM-00006,"Principles of general, organic, & biological c...",Janice Smith Gorzynski,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,"principles of general, organic, & biological c...",janice smith gorzynski,False
6,MM-00007,Foundations of Financial Management,Geoffrey A. Hirt; Bartley Danielsen; Stanley B...,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,foundations of financial management,geoffrey a hirt; bartley danielsen; stanley b ...,False
7,MM-00008,Adam Smith and the Death of David Hume,Rasmussen Dennis C,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,adam smith and the death of david hume,rasmussen dennis c,False
8,MM-00009,Quantum,Couture Ivan,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,quantum,couture ivan,False
9,MM-00010,Law of the land,Roger A. McEowen; Sharon E. Magee-Minor,0,mismatch_buku,NEEDS_MANUAL_VERIFICATION,,law of the land,roger a mceowen; sharon e magee-minor,False


## 6. Jika tersedia, bandingkan dengan mapping dari `dataset_v2`

Langkah ini bersifat opsional. Mapping lama dapat membantu menelusuri pasangan sumber, tetapi **mapping lama tidak otomatis dianggap benar**. Notebook hanya menampilkan struktur dan sampel; tidak mengubah label berdasarkan mapping tersebut.

In [7]:
mapping_path = PROJECT_ROOT / "dataset_v2" / "mapping_mismatch_buku.csv"

if mapping_path.exists():
    mapping = pd.read_csv(mapping_path, encoding="utf-8-sig", keep_default_na=False)
    print("Mapping ditemukan:", mapping.shape)
    print("Kolom mapping:", mapping.columns.tolist())
    display(mapping.head(10))
else:
    mapping = None
    print("mapping_mismatch_buku.csv tidak ditemukan. Lewati tahap ini.")

Mapping ditemukan: (1713, 6)
Kolom mapping: ['mismatch_id', 'valid_id', 'mapping_method', 'title_similarity', 'author_similarity', 'split']


,mismatch_id,valid_id,mapping_method,title_similarity,author_similarity,split
0,0,11221,exact_normalized_title,1.0,0.818182,latih
1,1,7542,exact_normalized_title,1.0,0.555556,latih
2,2,8437,exact_normalized_title,1.0,0.500000,latih
3,3,10159,exact_normalized_title,1.0,0.529412,uji
4,4,11290,exact_normalized_title,1.0,0.666667,latih
5,5,11102,exact_normalized_title,1.0,0.727273,latih
6,6,7122,exact_normalized_title,1.0,0.660377,latih
7,7,8066,exact_normalized_title,1.0,0.473684,latih
8,8,11097,exact_normalized_title,1.0,0.583333,latih
9,9,9932,exact_normalized_title,1.0,0.538462,latih


## 7. Ekspor file staging dan review queue

Output disimpan di `dataset_v3_staging/`. File sumber dan `dataset_v2` tidak disentuh.

Output:
- `valid_books_cleaned_staging.csv`: buku valid setelah pembersihan konservatif.
- `mismatch_books_review_queue.csv`: kandidat mismatch yang masih perlu verifikasi.
- `valid_books_quality_audit.csv`: seluruh data valid beserta penanda audit.
- `dataset_quality_summary.json`: ringkasan jumlah data.

**Jangan langsung menggunakan review queue sebagai label halu untuk training.** Selesaikan pemeriksaan label terlebih dahulu.

In [8]:
# Kolom bantu audit dipertahankan pada file audit,
# sedangkan staging valid menyimpan kolom sumber + status/jenis/review.
valid_audit_path = OUTPUT_DIR / "valid_books_quality_audit.csv"
valid_staging_path = OUTPUT_DIR / "valid_books_cleaned_staging.csv"
mismatch_queue_path = OUTPUT_DIR / "mismatch_books_review_queue.csv"

valid.to_csv(valid_audit_path, index=False, encoding="utf-8-sig")

# Hapus kolom internal key/flag dari staging training.
internal_cols = [
    "title_key", "author_key", "judul_kosong", "author_kosong",
    "duplikat_pasangan_teks", "judul_sama_author_berbeda",
    "author_duplikat_dalam_sel"
]
valid_staging = valid_clean.drop(
    columns=[c for c in internal_cols if c in valid_clean.columns]
)
valid_staging.to_csv(valid_staging_path, index=False, encoding="utf-8-sig")

mismatch_export = mismatch_review.drop(
    columns=["title_key", "author_key"]
)
mismatch_export.to_csv(
    mismatch_queue_path, index=False, encoding="utf-8-sig"
)

summary = {
    "valid_source_rows": int(len(valid_raw)),
    "valid_staging_rows": int(len(valid_staging)),
    "valid_rows_removed_conservatively": int(len(valid_raw) - len(valid_staging)),
    "valid_blank_titles": int(audit_summary["judul_kosong"]),
    "valid_blank_authors": int(audit_summary["author_kosong"]),
    "valid_duplicate_text_pairs": int(audit_summary["duplikat_pasangan_teks"]),
    "valid_same_title_different_author": int(audit_summary["judul_sama_author_berbeda"]),
    "valid_repeated_authors_in_cell": int(audit_summary["author_duplikat_dalam_sel"]),
    "mismatch_source_rows": int(len(mismatch_raw)),
    "mismatch_review_status": "NEEDS_MANUAL_VERIFICATION",
    "warning": "Do not train on mismatch review queue until labels are reviewed."
}

summary_path = OUTPUT_DIR / "dataset_quality_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Berhasil mengekspor:")
print(valid_audit_path)
print(valid_staging_path)
print(mismatch_queue_path)
print(summary_path)

display(pd.DataFrame(
    [{"output": p.name, "bytes": p.stat().st_size}
     for p in [valid_audit_path, valid_staging_path, mismatch_queue_path, summary_path]]
))

Berhasil mengekspor:
d:\capstone\civa\dataset_v3_staging\valid_books_quality_audit.csv
d:\capstone\civa\dataset_v3_staging\valid_books_cleaned_staging.csv
d:\capstone\civa\dataset_v3_staging\mismatch_books_review_queue.csv
d:\capstone\civa\dataset_v3_staging\dataset_quality_summary.json


,output,bytes
0,valid_books_quality_audit.csv,1541048
1,valid_books_cleaned_staging.csv,1190359
2,mismatch_books_review_queue.csv,220719
3,dataset_quality_summary.json,459


## 8. Membentuk dataset buku final setelah review

Jalankan bagian ini **hanya setelah** `mismatch_books_review_queue.csv` diperiksa dan kolom `review_status` diperbarui.

- `CONFIRMED_HALU` akan diberi `status=0`.
- `CONFIRMED_VALID` akan diberi `status=1`.
- `REJECT_MAPPING` dan `NEEDS_MANUAL_VERIFICATION` tidak dimasukkan.
- Baris valid dari sumber tetap `status=1`.

File yang dihasilkan masih berupa dataset buku. Penggabungan dengan dataset lama CIVA, pembuatan `grup_id`, serta split train/test harus dilakukan pada tahap terpisah dengan aturan group split agar turunan satu referensi tidak bocor ke dua split.

In [9]:
# Baca ulang file review queue yang mungkin sudah diedit secara manual.
reviewed_path = mismatch_queue_path

reviewed = pd.read_csv(
    reviewed_path, encoding="utf-8-sig", keep_default_na=False
)

allowed_statuses = {
    "CONFIRMED_HALU",
    "CONFIRMED_VALID",
    "REJECT_MAPPING",
    "NEEDS_MANUAL_VERIFICATION",
}

unknown = set(reviewed["review_status"].unique()) - allowed_statuses
if unknown:
    raise ValueError(f"Nilai review_status tidak dikenal: {unknown}")

confirmed = reviewed[
    reviewed["review_status"].isin(["CONFIRMED_HALU", "CONFIRMED_VALID"])
].copy()

confirmed["status"] = confirmed["review_status"].map({
    "CONFIRMED_HALU": 0,
    "CONFIRMED_VALID": 1
})

confirmed["jenis"] = np.where(
    confirmed["status"] == 0,
    "mismatch_buku",
    "asli"
)

# Dataset valid staging + mismatch yang sudah dikonfirmasi.
valid_for_final = valid_staging.copy()
valid_for_final["status"] = 1

# Samakan kolom agar concat aman; kolom sumber yang tidak tersedia pada mismatch akan kosong.
all_columns = sorted(set(valid_for_final.columns) | set(confirmed.columns))
book_dataset_reviewed = pd.concat(
    [
        valid_for_final.reindex(columns=all_columns),
        confirmed.reindex(columns=all_columns)
    ],
    ignore_index=True
)

final_book_path = OUTPUT_DIR / "dataset_buku_reviewed.csv"
book_dataset_reviewed.to_csv(
    final_book_path, index=False, encoding="utf-8-sig"
)

print("Baris valid sumber:", len(valid_for_final))
print("Baris mismatch terkonfirmasi:", int((confirmed["status"] == 0).sum()))
print("Baris kandidat yang dikonfirmasi valid:", int((confirmed["status"] == 1).sum()))
print("Baris belum diputuskan / ditolak:",
      int((~reviewed["review_status"].isin(["CONFIRMED_HALU", "CONFIRMED_VALID"])).sum()))
print("Total dataset buku reviewed:", len(book_dataset_reviewed))
print("Output:", final_book_path)

display(book_dataset_reviewed["status"].value_counts(dropna=False))

Baris valid sumber: 5894
Baris mismatch terkonfirmasi: 0
Baris kandidat yang dikonfirmasi valid: 0
Baris belum diputuskan / ditolak: 1713
Total dataset buku reviewed: 5894
Output: d:\capstone\civa\dataset_v3_staging\dataset_buku_reviewed.csv


status
1    5894
Name: count, dtype: int64

## 9. Checklist sebelum dataset dipakai training

Pastikan:
- [ ] Jumlah baris sumber dan hasil staging tercatat.
- [ ] Baris yang dihapus hanya duplikasi pasangan teks identik atau judul kosong sesuai aturan konservatif.
- [ ] Variasi urutan nama penulis tidak otomatis diubah menjadi label halu.
- [ ] Setiap mismatch yang digunakan sebagai `status=0` sudah dikonfirmasi.
- [ ] Data valid dan mismatch memiliki `grup_id` yang benar.
- [ ] Semua baris dari satu sumber dan turunannya berada pada split yang sama.
- [ ] Test set tidak digunakan untuk memilih threshold atau hyperparameter.

**Catatan metodologis:** notebook ini memperbaiki dan menyiapkan kualitas data buku. Ia tidak membuktikan kebenaran bibliografis secara otomatis dan tidak menjamin accuracy model akan naik. Dampaknya harus diuji dengan eksperimen training dan holdout yang terpisah.